# ⚙️ Vigyan-3B Engineering: Advanced Signal Processing & Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyansh001boy-tech/vigyan-7b-demo/blob/main/qwen_stem_demos/3b_engineering_colab.ipynb)
&nbsp;[![License: CC BY-NC 4.0](https://img.shields.io/badge/License-CC_BY--NC_4.0-lightgrey.svg)](https://creativecommons.org/licenses/by-nc/4.0/)
&nbsp;[![Hugging Face Adapter](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Adapter-yellow)](https://huggingface.co/shreyansh12183/vigyan-3b-adapter-engineering)

---

### 🌟 Model Overview
Specialized 3B Language Model calibrated for **Advanced Electrical & Mechanical Engineering**.
- **Base Model:** `Qwen/Qwen2.5-3B-Instruct`
- **Domain Adapter:** `shreyansh12183/vigyan-3b-adapter-engineering` (100% Public)
- **Hardware Requirement:** Free Google Colab **Tesla T4 GPU** (Allocates in **~2.5 GB VRAM** in 4-bit NF4).
- **License:** CC BY-NC 4.0 (Free for academic, student, and non-commercial research use).

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes gradio
print("✅ Runtime packages installed.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_ID = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_ID = "shreyansh12183/vigyan-3b-adapter-engineering"

print(f"⚡ PyTorch: {torch.__version__} | CUDA: {torch.cuda.get_device_name(0)}")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(BASE_ID, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, ADAPTER_ID).eval()
print("✓ Vigyan-3B Model loaded in VRAM (~2.5 GB)!")

In [ ]:
import gradio as gr

def solve(query, history):
    prompt = f"<|im_start|>user\n{query}\nProvide rigorous step-by-step reasoning with formal scientific derivations.<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=450, temperature=0.2, do_sample=True)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

demo = gr.ChatInterface(fn=solve, title="⚙️ Vigyan-3B Engineering: Advanced Signal Processing & Dynamics", description="Qwen 2.5 3B Architecture | Created by Shreyansh Singh", examples=["Derive the transfer function H(s) = V_out(s) / V_in(s) for a second-order Sallen-Key low-pass filter circuit.", "Calculate the state-space representation [A, B, C, D] for an inverted pendulum on a moving cart.", "Explain how Fast Fourier Transform (FFT) reduces discrete Fourier transform computational complexity from O(N^2) to O(N log N)."])
demo.launch(share=True)